# 1. Context

This notebook assesses reduction in WER in samples where WER >= 10%. This helps in identifying Model & Prompt Combination which could help in Post OCR Correction

# 2. Assumption

* TextBlock where correction is required is known apriori
   
   * This can be assessed via confidence value return by Surya OCR


# 3. Imports

In [ ]:
import pandas as pd
import jiwer
from pathlib import Path
from collections import defaultdict
import sys

In [ ]:
notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent.parent))

In [ ]:
from src.evaluation.metrics import cer, wer
from src.common.constants import language_to_writing_system, writing_system_to_language
from src.common.utils import get_script_results

# 4. Loading Consolidated Results

## 4.1. Loading Results

In [ ]:
# get csv paths for all language
results_root = Path("../../results/surya_ocr")
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [ ]:
writing_sys_dict = defaultdict(list)
for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)

script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [ ]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result, result_root=results_root)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

## 4.2. Adding CER and WER

In [ ]:
# computing cer & wer
gt_col = 'ground_truth'
ocr_output_cols = ['ocr_output_L_0', 'ocr_output_L_1', 'ocr_output_L_2', 'ocr_output_L_3']
cols_create_cer = ['cer_l0', 'cer_l1', 'cer_l2', 'cer_l3']
cols_create_wer = ['wer_l0', 'wer_l1', 'wer_l2', 'wer_l3']
for ocr_output_lvl, col_crt_cer in dict(zip(ocr_output_cols, cols_create_cer)).items():
    consolidated_df[col_crt_cer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: cer(x[gt_col], x[ocr_output_lvl]), axis=1)
for ocr_output_lvl, col_crt_wer in dict(zip(ocr_output_cols, cols_create_wer)).items():
    consolidated_df[col_crt_wer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: wer(x[gt_col], x[ocr_output_lvl]), axis=1)

In [ ]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

# 5. Selecting Text Blocks For Analysis

In [ ]:
_LANGUGAE_EXCLUDE_ = ["santali", "manipuri"]
results_lang_scoped = consolidated_df[~consolidated_df['LANGUAGE'].isin(_LANGUGAE_EXCLUDE_)]

In [ ]:
# degradation level & ocr output col map
degradation_ocr_col_map = {
    'L0': 'ocr_output_L_0',
    'L1': 'ocr_output_L_1',
    'L2': 'ocr_output_L_2',
    'L3': 'ocr_output_L_3'
}

In [ ]:
filtered_rows_list = []
common_cols = ['FILE_ID', 'LANGUAGE', 'GROUND_TRUTH']
WER_COLS = ['WER_L0', 'WER_L1', 'WER_L2', 'WER_L3']
CER_COLS = ['CER_L0', 'CER_L1', 'CER_L2', 'CER_L3']
for idx, row in results_lang_scoped.iterrows():
    for CER in CER_COLS:
        if row[CER] >= 0.1:
            degradation_id = CER.split('_')[-1]
            ocr_output_col = degradation_ocr_col_map[degradation_id].upper()
            selected_values = row[common_cols + [ocr_output_col] + [CER]]
            selected_values.rename({ocr_output_col: 'OCR_OUTPUT', CER: 'CER'}, inplace=True)
            selected_values['HIGH_CER'] = True
            filtered_rows_list.append(selected_values)
    # for WER in WER_COLS:
    #     if row[WER] >= 0.1:
    #         degradation_id = WER.split('_')[-1]
    #         ocr_output_col = degradation_ocr_col_map[degradation_id].upper()
    #         selected_values = row[common_cols + [ocr_output_col] + [WER]]
    #         selected_values.rename({ocr_output_col: 'OCR_OUTPUT', WER: 'WER'}, inplace=True)
    #         selected_values['HIGH_WER'] = True
    #         filtered_rows_list.append(selected_values)
    

In [ ]:
df_sample = pd.DataFrame(filtered_rows_list)

In [ ]:
df_sample.loc[df_sample['LANGUAGE'] == "hindi"].iloc[0].GROUND_TRUTH

In [ ]:
df_sample.loc[df_sample['LANGUAGE'] == "hindi"].iloc[0].OCR_OUTPUT

In [ ]:
df_sample.loc[df_sample['LANGUAGE'] == "hindi"].iloc[0]

In [ ]:
results_lang_scoped.loc[results_lang_scoped['LANGUAGE'] == "hindi"].iloc[0]